# Flash Drought Indicators

This notebook will introduce the two remote sensing datasets that we will be using as flash drought indicators in this course:
* Solar-Induced Fluorescence (SIF) data in the form of the [GOSIF dataset](https://globalecology.unh.edu/data/GOSIF.html)
* Root-zone Soil Moisture (SM) data from the [SMAP L4 Global 3-hourly 9 km dataset](https://search.earthdata.nasa.gov/search/granules/collection-details?p=C3480440870-NSIDC_CPRD)
* Soil Water Deficit Index (SWDI) derived from ERA-5 Land

### Why are we using these datasets?

The motivation for using SIF and SM data for detecting flash drought comes from an "impact-centric framework" developed by Koushan Mohammadi and Guiling Wang of the University of Connecticut. \[[3](https://doi.org/10.1073/pnas.2202767119)\]\[[4](https://doi.org/10.1175/BAMS-D-24-0143.1)\] Traditionally, drought was seen as a primarily climatological phenomenon where a deficit in precipitation and higher than normal temperatures were the key metrics used for detection \[[1](https://doi.org/10.46275/JOASC.2021.02.001)\]\[[2](https://doi.org/10.1002/wat2.1714)\]. Agricultural droughts are, however, assessed in terms of their impacts on cropland above any other factors, so the determination as to whether a precipitation deficit is material to crop growth depends on the crop type, existing soil moisture, and the climate of the growing region being studied. It is far more effective, therefore, to use land-process based metrics like SIF and SM to detect flash drought than it is to use metrics like temperature and precipitation. 

## Solar-Induced Chlorophyll Flourescence

SIF data can be thought of as a measure of the photosynthetic activity of plants in aggregate. The ARSET program has produced several courses on this data if you would like to learn more about this exciting measurement: [Solar Induced Fluorescence (SIF) Observations for Assessing Vegetation Changes Related to Floods, Drought, and Fire Impacts](https://www.earthdata.nasa.gov/learn/trainings/solar-induced-fluorescence-sif-observations-assessing-vegetation-changes-related). As most instruments capable of measuring SIF data from space right now are limited in terms of their spatial coverage and observational record, we will be relying on a gap-filled dataset called [GOSIF](https://globalecology.unh.edu/data/GOSIF.html) produced by Xing Li and Jingfeng Xiao of the University of New Hampshire (UNH). \[[10](https://doi.org/10.3390/rs11050517)\] Gap-filled data like GOSIF combines OCO-2 observations with a secondary source like Terra and Aqua MODIS to both fill in spatial gaps in the data and allow the SIF measurement to be extrapolated past the time range of the OCO-2 mission, which launched in 2014. Mohammadi and Wang use the GOME-2 and Contiguous SIF (CSIF) products in their own research as data sources, but GOSIF offers the same 8-day temporal resolution used in \[[4](https://doi.org/10.1175/BAMS-D-24-0143.1)\] and has publicly accessible data from 2000-2024.

In this first section, we will download a time range of GOSIF data from the UNH repository and unpack it into GeoTIFF format.

In [ ]:

from datetime import datetime
import os
from tqdm.notebook import tqdm

from download import download_unpack_gosif

In [ ]:
year = 2017
doy_range = range(73, 298, 8)

output_dir = "data/gosif"
os.makedirs(output_dir, exist_ok=True)

gosif_geotiffs: list[str] = []
for doy in tqdm(doy_range, desc="Downloading granules"):
    fname = download_unpack_gosif(
        year,
        day=doy,
        output_dir=output_dir,
        verbose=False
    )
    if fname:
        gosif_geotiffs.append(fname)

### Creating a Time Series of the SIF Data

Now that we've retrieved our data, we can create a time series of the average SIF value over the region of interest and plot it. In our first case study, we will use the Northern Great Plains region of the US defined in the Mohammadi and Wang paper as 45.00°-50.00°N and 106.00°-111°W \[[4](https://doi.org/10.1175/BAMS-D-24-0143.1)\]. We will use the rasterio library to perform this calculation.

In [ ]:
import csv

import numpy as np
import rasterio
from rasterio.windows import from_bounds

# Set the filename to use for the time series
time_series_fname = "northern_great_plains_2017_sif.csv"

# Northern Great Plains region of interest
# 45.00°–50.00°N, 106.00°–111.00°W
west, south, east, north = -111.0, 45.0, -106.0, 50.0

# The threshold and scale factor parameters come from the documentation: https://data.globalecology.unh.edu/data/GOSIF_v2/Fair_Data_Use_Policy_and_Readme_GOSIF_v2.pdf
# 32767 = water bodies, 32766 = ice/snow
gosif_data_thresh = 32765
# This value tells our code the conversion between pixel values in the GeoTIFF images to units of W/m^2/sr/μm
gosif_scale_factor = 0.0001

dates: list[datetime] = []
sif_means: list[float] = []
with rasterio.open(gosif_geotiffs[0]) as src:
    window = from_bounds(west, south, east, north, src.transform)
for geotiff in gosif_geotiffs:
    # Parse the date from the filename, e.g. GOSIF_2017073.tif = DOY 73
    doy = int(os.path.splitext(os.path.basename(geotiff))[0][-3:])
    dates.append(datetime.strptime(f"{year}{doy:03d}", "%Y%j"))

    with rasterio.open(geotiff) as src:
        data = src.read(1, window=window).astype(float)
        data[data > gosif_data_thresh] = np.nan
        sif_means.append(float(np.nanmean(data)) * gosif_scale_factor)

print(f"Computed spatial mean for {len(dates)} granules")

csv_path = os.path.join("data", time_series_fname)
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["date", "sif_mean"])
    # We have 4 sig figs from the source data
    writer.writerows(zip([d.strftime("%Y-%m-%d") for d in dates], [f"{sm:.4f}" for sm in sif_means]))

print(f"Saved to {csv_path}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

mean_fname = "northern_great_plains_mean_sif.csv"
doy_range = list(range(73, 298, 8))

# Read the saved CSV
plot_dates = []
plot_sif: list[float] = []
with open(csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        plot_dates.append(datetime.strptime(row["date"], "%Y-%m-%d"))
        plot_sif.append(float(row["sif_mean"]))

plot_mean_sif: list[float] = []
with open(mean_fname, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if int(row["doy"]) not in doy_range:
            continue
        plot_mean_sif.append(float(row["mean_sif"]))


# Plot the time series
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(plot_dates, plot_sif, marker="o", color="darkgreen", linewidth=1.5, markersize=4)
ax.plot(plot_dates, plot_mean_sif, color=(0, 0, 0.5), linewidth=1.5)
ax.set_ylabel("GOSIF (W/m$^2$/sr/μm)")
ax.set_title(f"GOSIF over Northern Great Plains ({year})\n45°–50°N, 106°–111°W")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
